Configuración, panel y utilidades

In [33]:
import cv2 as cv
import numpy as np
import sys, os, unicodedata

RUTA_IMAGEN = 'Imagen2.jpg'

def tamano_pantalla():
    try:
        import tkinter as tk
        r = tk.Tk(); r.withdraw()
        w, h = r.winfo_screenwidth(), r.winfo_screenheight()
        r.destroy()
        return int(w), int(h)
    except Exception:
        return 1366, 768

PANT_W, PANT_H = tamano_pantalla()
ANCHO_PANEL = 460


# Carga de la imagen, reducida si no cabe en la pantalla
imagen_original = cv.imread(RUTA_IMAGEN)
if imagen_original is None:
    sys.exit(f"Error: No se pudo cargar la imagen en '{RUTA_IMAGEN}'.")

max_w = (PANT_W - ANCHO_PANEL - 90) // 2      # caben 2 imágenes + el panel
max_h = PANT_H - 170
e = min(max_w / imagen_original.shape[1], max_h / imagen_original.shape[0], 1.0)
imagen = (cv.resize(imagen_original, None, fx=e, fy=e, interpolation=cv.INTER_AREA)
          if e < 1.0 else imagen_original.copy())

alto, ancho = imagen.shape[:2]
size = (ancho, alto)


PAL = dict(fondo_a=(46, 18, 76), fondo_b=(20, 8, 38), tarjeta=(66, 34, 108),
           pista=(96, 64, 140), acento=(168, 85, 247), acento2=(217, 70, 239),
           texto=(245, 235, 255), suave=(196, 170, 232), blanco=(255, 255, 255))

def B(c):
    return (int(c[2]), int(c[1]), int(c[0]))

def rect_redondeado(img, p1, p2, r, color):
    (x1, y1), (x2, y2) = p1, p2
    r = max(1, min(r, (x2 - x1) // 2, (y2 - y1) // 2))
    cv.rectangle(img, (x1 + r, y1), (x2 - r, y2), color, -1, cv.LINE_AA)
    cv.rectangle(img, (x1, y1 + r), (x2, y2 - r), color, -1, cv.LINE_AA)
    for cx, cy in ((x1 + r, y1 + r), (x2 - r, y1 + r), (x1 + r, y2 - r), (x2 - r, y2 - r)):
        cv.circle(img, (cx, cy), r, color, -1, cv.LINE_AA)

def texto_img(img, s, org, escala=0.55, color=None):
    color = color or B(PAL['texto'])
    while escala > 0.3 and org[0] + cv.getTextSize(s, cv.FONT_HERSHEY_SIMPLEX, escala, 1)[0][0] > img.shape[1] - 6:
        escala -= 0.02
    cv.putText(img, s, org, cv.FONT_HERSHEY_SIMPLEX, escala, (20, 5, 30), 3, cv.LINE_AA)
    cv.putText(img, s, org, cv.FONT_HERSHEY_SIMPLEX, escala, color, 1, cv.LINE_AA)

try:
    from PIL import Image, ImageDraw, ImageFont
    import matplotlib
    _DIR = os.path.join(matplotlib.get_data_path(), 'fonts', 'ttf')
    _FUENTES = {'n': os.path.join(_DIR, 'DejaVuSans.ttf'),
                'b': os.path.join(_DIR, 'DejaVuSans-Bold.ttf'),
                'm': os.path.join(_DIR, 'DejaVuSansMono.ttf')}
    ImageFont.truetype(_FUENTES['n'], 12)
    HAY_PIL = True
except Exception:
    HAY_PIL = False

_cache_fuentes = {}
def _fuente(estilo, tam):
    clave = (estilo, int(tam))
    if clave not in _cache_fuentes:
        _cache_fuentes[clave] = ImageFont.truetype(_FUENTES[estilo], int(tam))
    return _cache_fuentes[clave]

def _pintar_textos(img, textos):
    if HAY_PIL:
        pil = Image.fromarray(cv.cvtColor(img, cv.COLOR_BGR2RGB))
        d = ImageDraw.Draw(pil)
        for s, x, y, tam, col, est, anc, maxw in textos:
            tam = int(tam); f = _fuente(est, tam)
            while maxw and f.getlength(s) > maxw and tam > 8:     # evita que se corte
                tam -= 1; f = _fuente(est, tam)
            d.text((x, y), s, font=f, fill=tuple(col), anchor='rs' if anc == 'r' else 'ls')
        return cv.cvtColor(np.array(pil), cv.COLOR_RGB2BGR)
    for s, x, y, tam, col, est, anc, maxw in textos:
        s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
        esc = tam / 30.0
        while maxw and cv.getTextSize(s, cv.FONT_HERSHEY_SIMPLEX, esc, 1)[0][0] > maxw and esc > 0.3:
            esc -= 0.02
        w = cv.getTextSize(s, cv.FONT_HERSHEY_SIMPLEX, esc, 1)[0][0]
        cv.putText(img, s, (x - w if anc == 'r' else x, y), cv.FONT_HERSHEY_SIMPLEX,
                   esc, B(col), 1, cv.LINE_AA)
    return img

def slider(n, etiqueta, vmin, vmax, ini, fmt=str):
    return dict(n=n, label=etiqueta, min=vmin, max=max(vmax, vmin + 1), ini=ini, fmt=fmt, switch=False)

def interruptor(n, etiqueta, ini=1, fmt=lambda v: 'Sí' if v else 'No'):
    return dict(n=n, label=etiqueta, min=0, max=1, ini=ini, fmt=fmt, switch=True)


class Panel:
    CAB, FILA, LIN, PAD = 64, 48, 21, 22

    def __init__(self, ventana, cabecera, controles, n_info=4):
        self.ventana, self.cabecera, self.ctrl, self.n_info = ventana, cabecera, controles, n_info
        self.vals = {c['n']: c['ini'] for c in controles}
        self.drag = None
        h0 = self.CAB + self.FILA * len(controles) + 22 + self.LIN * n_info + 56
        self.u = max(0.6, min(1.0, (PANT_H - 110) / h0))        # se adapta al alto de pantalla
        self.w, self.h = int(ANCHO_PANEL * self.u), int(h0 * self.u)
        t = np.linspace(0, 1, self.h).reshape(-1, 1, 1)
        grad = np.array(B(PAL['fondo_a'])) * (1 - t) + np.array(B(PAL['fondo_b'])) * t
        self.fondo = np.ascontiguousarray(np.broadcast_to(grad, (self.h, self.w, 3))).astype(np.uint8)
        cv.namedWindow(ventana, cv.WINDOW_AUTOSIZE)
        cv.setMouseCallback(ventana, self._raton)

    def _y(self, i):  return int((self.CAB + self.FILA * i) * self.u)
    def _x0(self):    return int(self.PAD * self.u)
    def _x1(self):    return self.w - int(self.PAD * self.u)

    def valor(self, n):  return self.vals[n]
    def reiniciar(self): self.vals = {c['n']: c['ini'] for c in self.ctrl}

    def _fijar_x(self, i, x):
        c = self.ctrl[i]; x0, x1 = self._x0(), self._x1()
        t = min(max((x - x0) / float(x1 - x0), 0.0), 1.0)
        self.vals[c['n']] = int(round(c['min'] + t * (c['max'] - c['min'])))

    def _raton(self, ev, x, y, flags, _):
        if ev in (cv.EVENT_LBUTTONDOWN, cv.EVENT_RBUTTONDOWN):
            for i, c in enumerate(self.ctrl):
                y0 = self._y(i)
                if y0 <= y < y0 + int(self.FILA * self.u):
                    if ev == cv.EVENT_RBUTTONDOWN:
                        self.vals[c['n']] = c['ini']
                    elif c['switch']:
                        self.vals[c['n']] = 1 - self.vals[c['n']]
                    else:
                        self.drag = i; self._fijar_x(i, x)
                    break
        elif ev == cv.EVENT_MOUSEMOVE and self.drag is not None:
            if flags & cv.EVENT_FLAG_LBUTTON: self._fijar_x(self.drag, x)
            else: self.drag = None
        elif ev == cv.EVENT_LBUTTONUP:
            self.drag = None

    def dibujar(self, lineas_info):
        u, img, textos = self.u, self.fondo.copy(), []
        x0, x1 = self._x0(), self._x1()
        T = lambda s, x, y, tam, col, est='n', anc='l', mw=None: textos.append((s, int(x), int(y), tam * u, col, est, anc, mw))

        T(self.cabecera, x0, 36 * u, 22, PAL['texto'], 'b', mw=x1 - x0)
        cv.line(img, (x0, int(48 * u)), (x0 + int(64 * u), int(48 * u)), B(PAL['acento2']), max(2, int(3 * u)), cv.LINE_AA)

        for i, c in enumerate(self.ctrl):
            y0, v = self._y(i), self.vals[c['n']]
            T(c['label'], x0, y0 + 20 * u, 14.5, PAL['suave'], mw=(x1 - x0) * 0.62)
            if c['switch']:
                pw, ph = int(46 * u), int(22 * u)
                px2, py1 = x1, y0 + int(6 * u)
                rect_redondeado(img, (px2 - pw, py1), (px2, py1 + ph), ph // 2,
                                B(PAL['acento'] if v else PAL['pista']))
                kx = px2 - ph // 2 - 1 if v else px2 - pw + ph // 2 + 1
                cv.circle(img, (kx, py1 + ph // 2), ph // 2 - 3, B(PAL['blanco']), -1, cv.LINE_AA)
                T(c['fmt'](v), px2 - pw - 10 * u, y0 + 20 * u, 14.5, PAL['texto'], 'b', 'r')
            else:
                T(c['fmt'](v), x1, y0 + 20 * u, 14.5, PAL['texto'], 'b', 'r')
                ty, grosor = y0 + int(36 * u), max(4, int(6 * u))
                t = (v - c['min']) / float(c['max'] - c['min'])
                xk = int(x0 + t * (x1 - x0))
                cv.line(img, (x0, ty), (x1, ty), B(PAL['pista']), grosor, cv.LINE_AA)
                for xx in (x0, x1): cv.circle(img, (xx, ty), grosor // 2, B(PAL['pista']), -1, cv.LINE_AA)
                origen = int(x0 + (-c['min'] / float(c['max'] - c['min'])) * (x1 - x0)) if c['min'] < 0 < c['max'] else x0
                cv.line(img, (origen, ty), (xk, ty), B(PAL['acento']), grosor, cv.LINE_AA)
                cv.circle(img, (xk, ty), int(9 * u), B(PAL['acento2']), -1, cv.LINE_AA)
                cv.circle(img, (xk, ty), int(5 * u), B(PAL['blanco']), -1, cv.LINE_AA)

        yi = self._y(len(self.ctrl)) + int(6 * u)
        yf = yi + int((self.LIN * self.n_info + 20) * u)
        rect_redondeado(img, (x0 - int(8 * u), yi), (x1 + int(8 * u), yf), int(12 * u), B(PAL['tarjeta']))
        for k, s in enumerate(lineas_info):
            T(s, x0 + 4 * u, yi + (24 + self.LIN * k) * u, 13, PAL['texto'], 'm', mw=(x1 - x0) - 8 * u)
        T('ESC salir  ·  R reiniciar  ·  clic dcho: restablecer control', x0, self.h - 16 * u, 11.5, PAL['suave'], mw=x1 - x0)
        return _pintar_textos(img, textos)

def colocar(panel, ventanas):
    x = 15
    for v in ventanas:
        cv.moveWindow(v, x, 30); x += ancho + 20
    cv.moveWindow(panel.ventana, max(x, PANT_W - panel.w - 15), 30)

def ventana_cerrada(nombre):
    try:
        return cv.getWindowProperty(nombre, cv.WND_PROP_VISIBLE) < 1
    except cv.error:
        return True

def cerrar_ventanas():
    cv.destroyAllWindows()
    cv.waitKey(1)

print(f"Imagen: {ancho}x{alto} (pantalla {PANT_W}x{PANT_H}) | fuente TrueType: {HAY_PIL}")

Imagen: 685x514 (pantalla 1920x1080) | fuente TrueType: True


Apartado 1a: traslación, rotación y escalado

In [34]:
# Orden: 1) Escalado  2) Rotación  3) Traslación
#   M = T(tx,ty) @ [T(c_giro) @ R(θ) @ T(-c_giro)] @ [T(c_esc) @ S(sx,sy) @ T(-c_esc)]

def mat_traslacion(tx, ty):
    return np.float32([[1, 0, tx],
                       [0, 1, ty],
                       [0, 0,  1]])

def mat_rotacion(angulo_grados, cx, cy):
    """Rotación alrededor de (cx, cy). Positivo = antihorario."""
    t = np.radians(angulo_grados)
    c, s = np.cos(t), np.sin(t)
    R = np.float32([[ c, s, 0],      # eje Y hacia abajo -> signos como en los ejemplos
                    [-s, c, 0],
                    [ 0, 0, 1]])
    return mat_traslacion(cx, cy) @ R @ mat_traslacion(-cx, -cy)

def mat_escalado(sx, sy, cx, cy):
    """Escalado (sx, sy) respecto al punto (cx, cy), que queda fijo."""
    S = np.float32([[sx,  0, 0],
                    [ 0, sy, 0],
                    [ 0,  0, 1]])
    return mat_traslacion(cx, cy) @ S @ mat_traslacion(-cx, -cy)

V_ORIG, V_RES = 'Original (Clic izq: giro | Clic der: escala)', 'Resultado (Arrastrar para mover)'
px = lambda v: f'{v:+d} px'

panel_raton = Panel('Controles Ratón', 'Transformaciones 2D + Ratón', [
    slider('tx',  'Traslación X',            -ancho, ancho, 0, px),
    slider('ty',  'Traslación Y',            -alto,  alto,  0, px),
    slider('ang', 'Ángulo de giro',          -180,   180,   0, lambda v: f'{v:+d}°'),
    slider('cgx', 'Centro de giro X',        0, ancho, ancho // 2, lambda v: f'{v} px'),
    slider('cgy', 'Centro de giro Y',        0, alto,  alto // 2,  lambda v: f'{v} px'),
    slider('sx',  'Factor de escala X',      1, 400, 100, lambda v: f'{v / 100:.2f}×'),
    slider('sy',  'Factor de escala Y',      1, 400, 100, lambda v: f'{v / 100:.2f}×'),
    interruptor('uni', 'Escalado uniforme',  1),
    slider('csx', 'Centro de escalado X',    0, ancho, ancho // 2, lambda v: f'{v} px'),
    slider('csy', 'Centro de escalado Y',    0, alto,  alto // 2,  lambda v: f'{v} px'),
], n_info=5)

for v in (V_ORIG, V_RES):
    cv.namedWindow(v, cv.WINDOW_AUTOSIZE)
colocar(panel_raton, [V_ORIG, V_RES])

# Variables de arrastre para V_RES
arrastrando = False
punto_inicio = (0, 0)
tx_base, ty_base = 0, 0

def raton_original(ev, x, y, flags, param):
    if ev == cv.EVENT_LBUTTONDOWN:       # Centro de giro
        panel_raton.vals['cgx'] = max(0, min(ancho, x))
        panel_raton.vals['cgy'] = max(0, min(alto, y))
    elif ev == cv.EVENT_RBUTTONDOWN:     # Centro de escalado
        panel_raton.vals['csx'] = max(0, min(ancho, x))
        panel_raton.vals['csy'] = max(0, min(alto, y))

def raton_resultado(ev, x, y, flags, param):
    global arrastrando, punto_inicio, tx_base, ty_base
    if ev == cv.EVENT_LBUTTONDOWN:
        arrastrando = True
        punto_inicio = (x, y)
        tx_base = panel_raton.valor('tx')
        ty_base = panel_raton.valor('ty')
    elif ev == cv.EVENT_MOUSEMOVE and arrastrando:
        dx = x - punto_inicio[0]
        dy = y - punto_inicio[1]
        panel_raton.vals['tx'] = max(-ancho, min(ancho, tx_base + dx))
        panel_raton.vals['ty'] = max(-alto, min(alto, ty_base + dy))
    elif ev == cv.EVENT_LBUTTONUP:
        arrastrando = False

cv.setMouseCallback(V_ORIG, raton_original)
cv.setMouseCallback(V_RES, raton_resultado)

try:
    while True:
        g = panel_raton.valor
        tx, ty, angulo = g('tx'), g('ty'), g('ang')
        cgx, cgy, csx, csy = g('cgx'), g('cgy'), g('csx'), g('csy')
        sx = g('sx') / 100.0
        sy = sx if g('uni') == 1 else g('sy') / 100.0

        M = mat_traslacion(tx, ty) @ mat_rotacion(angulo, cgx, cgy) @ mat_escalado(sx, sy, csx, csy)
        res = cv.warpAffine(imagen, M[:2, :], size)

        vista = imagen.copy()
        for (x, y), col, forma in (((cgx, cgy), PAL['acento2'], cv.MARKER_CROSS),
                                   ((csx, csy), PAL['suave'],   cv.MARKER_TILTED_CROSS)):
            cv.drawMarker(vista, (x, y), (20, 5, 30), forma, 30, 5)
            cv.drawMarker(vista, (x, y), B(col), forma, 30, 2)
            cv.circle(vista, (x, y), 9, (20, 5, 30), 4, cv.LINE_AA)
            cv.circle(vista, (x, y), 9, B(col), 2, cv.LINE_AA)

        texto_img(vista, 'Clic izq: giro | Clic dcho: escala', (10, 25), 0.5)

        cv.imshow(V_ORIG, vista)
        cv.imshow(V_RES, res)
        cv.imshow(panel_raton.ventana, panel_raton.dibujar([
            f"t=({tx},{ty})  θ={angulo}°  s=({sx:.2f},{sy:.2f})",
            "M (3x3):",
            *[" ".join(f"{v:9.2f}" for v in fila) for fila in M],
        ]))

        tecla = cv.waitKey(20) & 0xFF
        if tecla == 27 or ventana_cerrada(panel_raton.ventana):
            break
        if tecla in (ord('r'), ord('R')):
            panel_raton.reiniciar()
finally:
    cerrar_ventanas()


Apartado Opcional: Traslaciones y rotación sobre video

In [35]:
# Transformaciones sobre vídeo en tiempo real
cap = cv.VideoCapture('video.mp4') # 0 para webcam, o sustituir por 'video.mp4'

if not cap.isOpened():
    print("No se pudo acceder a la cámara.")
else:
    ret, frame0 = cap.read()
    if ret:
        e_v = min(max_w / frame0.shape[1], max_h / frame0.shape[0], 1.0)
        h_v, w_v = int(frame0.shape[0] * e_v), int(frame0.shape[1] * e_v)
        size_v = (w_v, h_v)

        panel_vid = Panel('Controles Video', 'Transformaciones en Video', [
            slider('tx',  'Traslación X',       -w_v, w_v, 0, px),
            slider('ty',  'Traslación Y',       -h_v, h_v, 0, px),
            slider('ang', 'Ángulo de giro',     -180, 180, 0, lambda v: f'{v:+d}°'),
            slider('sx',  'Escala',             10, 300, 100, lambda v: f'{v / 100:.2f}×'),
        ], n_info=3)

        V_VID_O, V_VID_R = 'Video Entrada', 'Video Transformado'
        for v in (V_VID_O, V_VID_R): cv.namedWindow(v, cv.WINDOW_AUTOSIZE)
        colocar(panel_vid, [V_VID_O, V_VID_R])

        try:
            while True:
                ret, frame = cap.read()
                if not ret: break
                frame = cv.resize(frame, size_v, interpolation=cv.INTER_AREA)

                tx, ty = panel_vid.valor('tx'), panel_vid.valor('ty')
                ang = panel_vid.valor('ang')
                s = panel_vid.valor('sx') / 100.0
                cx, cy = w_v // 2, h_v // 2

                M = mat_traslacion(tx, ty) @ mat_rotacion(ang, cx, cy) @ mat_escalado(s, s, cx, cy)
                res = cv.warpAffine(frame, M[:2, :], size_v)

                cv.imshow(V_VID_O, frame)
                cv.imshow(V_VID_R, res)
                cv.imshow(panel_vid.ventana, panel_vid.dibujar([
                    f"Frame: {w_v}x{h_v} px",
                    f"t=({tx},{ty})  θ={ang}°  s={s:.2f}x",
                    "Presione ESC para finalizar"
                ]))

                tecla = cv.waitKey(10) & 0xFF
                if tecla == 27 or ventana_cerrada(panel_vid.ventana): break
                if tecla in (ord('r'), ord('R')): panel_vid.reiniciar()
        finally:
            cap.release()
            cerrar_ventanas()

Apartado Opcional: Transformación Afín por 3 pares de puntos

In [36]:
pts_orig_3 = []
pts_dest_3 = []

V_AFIN_O, V_AFIN_D = 'Origen (3 clics)', 'Destino (3 clics y resultado)'
for v in (V_AFIN_O, V_AFIN_D): cv.namedWindow(v, cv.WINDOW_AUTOSIZE)

def click_afin_orig(ev, x, y, flags, param):
    if ev == cv.EVENT_LBUTTONDOWN and len(pts_orig_3) < 3:
        pts_orig_3.append((x, y))

def click_afin_dest(ev, x, y, flags, param):
    if ev == cv.EVENT_LBUTTONDOWN and len(pts_dest_3) < 3:
        pts_dest_3.append((x, y))

cv.setMouseCallback(V_AFIN_O, click_afin_orig)
cv.setMouseCallback(V_AFIN_D, click_afin_dest)

print("Haz clic en 3 puntos en 'Origen' y 3 puntos en 'Destino'. Presiona 'R' para reiniciar, 'ESC' para salir.")

try:
    while True:
        img_o = imagen.copy()
        img_d = np.full_like(imagen, B(PAL['fondo_b']))

        # Dibujar puntos origen
        for i, p in enumerate(pts_orig_3, 1):
            cv.circle(img_o, p, 6, B(PAL['acento2']), -1, cv.LINE_AA)
            texto_img(img_o, f"P{i}", (p[0] + 8, p[1] - 8), 0.5)

        # Si tenemos los 3 pares, calculamos y aplicamos la transformación
        if len(pts_orig_3) == 3 and len(pts_dest_3) == 3:
            M_afin = cv.getAffineTransform(np.float32(pts_orig_3), np.float32(pts_dest_3))
            img_d = cv.warpAffine(imagen, M_afin, size)

        # Dibujar puntos destino
        for i, p in enumerate(pts_dest_3, 1):
            cv.circle(img_d, p, 6, B(PAL['acento']), -1, cv.LINE_AA)
            texto_img(img_d, f"P{i}'", (p[0] + 8, p[1] - 8), 0.5)

        cv.imshow(V_AFIN_O, img_o)
        cv.imshow(V_AFIN_D, img_d)

        tecla = cv.waitKey(30) & 0xFF
        if tecla == 27: break
        if tecla in (ord('r'), ord('R')):
            pts_orig_3.clear()
            pts_dest_3.clear()
finally:
    cerrar_ventanas()

Haz clic en 3 puntos en 'Origen' y 3 puntos en 'Destino'. Presiona 'R' para reiniciar, 'ESC' para salir.


Apartado Opcional: Transformación Proyectiva por 4 pares de puntos (Homografía)

In [37]:
pts_orig_4 = []
pts_dest_4 = []

V_HOM_O, V_HOM_D = 'Origen Proyectivo (4 clics)', 'Destino Proyectivo (4 clics)'
for v in (V_HOM_O, V_HOM_D): cv.namedWindow(v, cv.WINDOW_AUTOSIZE)

def click_hom_orig(ev, x, y, flags, param):
    if ev == cv.EVENT_LBUTTONDOWN and len(pts_orig_4) < 4:
        pts_orig_4.append((x, y))

def click_hom_dest(ev, x, y, flags, param):
    if ev == cv.EVENT_LBUTTONDOWN and len(pts_dest_4) < 4:
        pts_dest_4.append((x, y))

cv.setMouseCallback(V_HOM_O, click_hom_orig)
cv.setMouseCallback(V_HOM_D, click_hom_dest)

try:
    while True:
        img_o = imagen.copy()
        img_d = np.full_like(imagen, B(PAL['fondo_b']))

        for i, p in enumerate(pts_orig_4, 1):
            cv.circle(img_o, p, 6, B(PAL['acento2']), -1, cv.LINE_AA)
            texto_img(img_o, f"{i}", (p[0] + 8, p[1] - 8), 0.5)

        if len(pts_orig_4) == 4 and len(pts_dest_4) == 4:
            try:
                H = cv.getPerspectiveTransform(np.float32(pts_orig_4), np.float32(pts_dest_4))
                img_d = cv.warpPerspective(imagen, H, size)
            except cv.error:
                pass

        for i, p in enumerate(pts_dest_4, 1):
            cv.circle(img_d, p, 6, B(PAL['acento']), -1, cv.LINE_AA)
            texto_img(img_d, f"{i}'", (p[0] + 8, p[1] - 8), 0.5)

        cv.imshow(V_HOM_O, img_o)
        cv.imshow(V_HOM_D, img_d)

        tecla = cv.waitKey(30) & 0xFF
        if tecla == 27: break
        if tecla in (ord('r'), ord('R')):
            pts_orig_4.clear()
            pts_dest_4.clear()
finally:
    cerrar_ventanas()

Apartado 1b: ventana de proyección

In [38]:
# Se traza un cuadrilátero (ventana) y se proyecta la imagen dentro de él
V_PROY, V_ORIG_B = 'Proyección', 'Imagen original'

# Origen: las 4 esquinas de la imagen (sup-izq, inf-izq, inf-der, sup-der)
pts_origen = np.float32([[0, 0], [0, alto], [ancho, alto], [ancho, 0]])

# Vértices iniciales de la ventana (el lado derecho se aleja)
ini = [(0.25, 0.20), (0.25, 0.80), (0.75, 0.65), (0.75, 0.35)]
nombres_v = ['sup. izq.', 'inf. izq.', 'inf. der.', 'sup. der.']

ctrls = []
for i, (nom, (fx, fy)) in enumerate(zip(nombres_v, ini), start=1):
    ctrls.append(slider(f'x{i}', f'Vértice {i} ({nom}) · X', 0, ancho, int(fx * ancho), lambda v: f'{v} px'))
    ctrls.append(slider(f'y{i}', f'Vértice {i} ({nom}) · Y', 0, alto,  int(fy * alto),  lambda v: f'{v} px'))
ctrls.append(interruptor('fondo', 'Mostrar imagen atenuada de fondo', 1))

panel = Panel('Controles 1b', 'Ventana de proyección', ctrls, n_info=4)

for v in (V_ORIG_B, V_PROY):
    cv.namedWindow(v, cv.WINDOW_AUTOSIZE)
colocar(panel, [V_ORIG_B, V_PROY])

mascara_total = np.full((alto, ancho), 255, np.uint8)   # silueta de la imagen completa
tinte = np.full_like(imagen, B(PAL['fondo_b']))
ultima, Tp = None, np.eye(3)

try:
    while True:
        # 1. Ventana de proyección (destino) definida por los sliders
        pts_destino = np.float32([[panel.valor(f'x{i}'), panel.valor(f'y{i}')] for i in range(1, 5)])

        try:
            # 2. Matriz proyectiva 3x3 y proyección de la imagen
            Tp = cv.getPerspectiveTransform(pts_origen, pts_destino)
            proyectada = cv.warpPerspective(imagen, Tp, size)
            mascara = cv.warpPerspective(mascara_total, Tp, size)

            # 3. Fondo y composición: la imagen solo aparece dentro de la ventana
            if panel.valor('fondo') == 1:
                lienzo = cv.addWeighted((imagen * 0.35).astype(np.uint8), 0.85, tinte, 0.15, 0)
            else:
                lienzo = tinte.copy()
            lienzo[mascara > 0] = proyectada[mascara > 0]

            # 4. Se traza la ventana: contorno + vértices numerados
            pts = pts_destino.astype(np.int32)
            cv.polylines(lienzo, [pts.reshape((-1, 1, 2))], True, B(PAL['acento2']), 3, cv.LINE_AA)
            for i, p in enumerate(pts, start=1):
                cv.circle(lienzo, tuple(int(c) for c in p), 7, B(PAL['blanco']), -1, cv.LINE_AA)
                cv.circle(lienzo, tuple(int(c) for c in p), 7, B(PAL['acento']), 2, cv.LINE_AA)
                texto_img(lienzo, str(i), (int(p[0]) + 10, int(p[1]) - 10), 0.6)
            ultima = lienzo
        except cv.error:
            pass    # vértices degenerados (colineales): se mantiene el último frame válido

        vista = imagen.copy()
        cv.rectangle(vista, (1, 1), (ancho - 2, alto - 2), B(PAL['acento2']), 3)
        cv.imshow(V_ORIG_B, vista)
        if ultima is not None:
            cv.imshow(V_PROY, ultima)
        cv.imshow(panel.ventana, panel.dibujar([
            "Matriz proyectiva Tp (3x3):",
            *[" ".join(f"{v:10.4f}" for v in fila) for fila in Tp],
        ]))

        tecla = cv.waitKey(20) & 0xFF
        if tecla == 27 or ventana_cerrada(panel.ventana):
            break
        if tecla in (ord('r'), ord('R')):
            panel.reiniciar()
finally:
    cerrar_ventanas()

Apartado Optativo: Imagen Especular mediante Matrices Homogéneas

In [39]:
def mat_especular(horizontal=True, vertical=False):
    sx = -1 if horizontal else 1
    sy = -1 if vertical else 1
    # Matriz base de reflexión
    S = np.float32([[sx,  0, 0],
                    [ 0, sy, 0],
                    [ 0,  0, 1]])
    # Trasladamos al centro de la imagen para reflejar sobre sus ejes medianos
    cx, cy = ancho / 2.0, alto / 2.0
    return mat_traslacion(cx, cy) @ S @ mat_traslacion(-cx, -cy)

panel_esp = Panel('Controles Especular', 'Imagen Especular', [
    interruptor('h', 'Reflexión Horizontal (Eje Y)', 1),
    interruptor('v', 'Reflexión Vertical (Eje X)', 0),
], n_info=5)

V_ESP = 'Resultado Especular'
cv.namedWindow(V_ESP, cv.WINDOW_AUTOSIZE)
colocar(panel_esp, [V_ESP])

try:
    while True:
        ref_h = bool(panel_esp.valor('h'))
        ref_v = bool(panel_esp.valor('v'))

        M_esp = mat_especular(ref_h, ref_v)
        res_esp = cv.warpAffine(imagen, M_esp[:2, :], size)

        cv.imshow(V_ESP, res_esp)
        cv.imshow(panel_esp.ventana, panel_esp.dibujar([
            f"Modo: {'H ' if ref_h else ''}{'V' if ref_v else ('Normal' if not ref_h else '')}",
            "Matriz de reflexión aplicada:",
            *[" ".join(f"{v:7.2f}" for v in fila) for fila in M_esp]
        ]))

        tecla = cv.waitKey(30) & 0xFF
        if tecla == 27 or ventana_cerrada(panel_esp.ventana): break
        if tecla in (ord('r'), ord('R')): panel_esp.reiniciar()
finally:
    cerrar_ventanas()

Apartado 1c: distorsión de lente

In [40]:
# Modelo [k1, k2, p1, p2, k3] (k = radiales, p = tangenciales)
#   k > 0 -> efecto barril | k < 0 -> efecto almohadilla
V_ORIG_C, V_DIST = 'Original', 'Distorsionada'

K_MAX = 1.0      # rango de k1, k2, k3: [-K_MAX, K_MAX]
P_MAX = 0.1      # rango de p1, p2:     [-P_MAX, P_MAX]

fk = lambda v: f'{v / 100 * K_MAX:+.2f}'
fp = lambda v: f'{v / 100 * P_MAX:+.3f}'

panel = Panel('Controles 1c', 'Distorsión de lente', [
    slider('k1', 'k1 · radial',      -100, 100, 0, fk),
    slider('k2', 'k2 · radial',      -100, 100, 0, fk),
    slider('k3', 'k3 · radial',      -100, 100, 0, fk),
    slider('p1', 'p1 · tangencial',  -100, 100, 0, fp),
    slider('p2', 'p2 · tangencial',  -100, 100, 0, fp),
    slider('f',  'Distancia focal f (fx = fy)', 1, 2 * ancho, ancho, lambda v: f'{v} px'),
    slider('cx', 'Centro óptico cx', 0, ancho, ancho // 2, lambda v: f'{v} px'),
    slider('cy', 'Centro óptico cy', 0, alto,  alto // 2,  lambda v: f'{v} px'),
    interruptor('rejilla', 'Mostrar rejilla', 1),
], n_info=3)


def con_rejilla(img, paso=None):
    paso = paso or max(16, ancho // 16)
    out = img.copy()
    for x in range(0, ancho, paso):
        cv.line(out, (x, 0), (x, alto), B(PAL['acento2']), 1, cv.LINE_AA)
    for y in range(0, alto, paso):
        cv.line(out, (0, y), (ancho, y), B(PAL['acento2']), 1, cv.LINE_AA)
    return out


def aplicar_distorsion(img, k1, k2, p1, p2, k3, f, cx, cy):
    dist_coeffs = np.array([k1, k2, p1, p2, k3], dtype=np.float64)
    matriz_camara = np.float64([[f, 0, cx],
                                [0, f, cy],
                                [0, 0,  1]])
    return cv.undistort(img, matriz_camara, dist_coeffs)


for v in (V_ORIG_C, V_DIST):
    cv.namedWindow(v, cv.WINDOW_AUTOSIZE)
colocar(panel, [V_ORIG_C, V_DIST])

try:
    while True:
        g = panel.valor
        k1, k2, k3 = g('k1') / 100 * K_MAX, g('k2') / 100 * K_MAX, g('k3') / 100 * K_MAX
        p1, p2 = g('p1') / 100 * P_MAX, g('p2') / 100 * P_MAX

        fuente = con_rejilla(imagen) if g('rejilla') == 1 else imagen
        distorsionada = aplicar_distorsion(fuente, k1, k2, p1, p2, k3, g('f'), g('cx'), g('cy'))

        cv.imshow(V_ORIG_C, fuente)
        cv.imshow(V_DIST, distorsionada)
        cv.imshow(panel.ventana, panel.dibujar([
            f"k1={k1:+.2f}   k2={k2:+.2f}   k3={k3:+.2f}",
            f"p1={p1:+.3f}  p2={p2:+.3f}",
            "k>0: barril   |   k<0: almohadilla",
        ]))

        tecla = cv.waitKey(20) & 0xFF
        if tecla == 27 or ventana_cerrada(panel.ventana):
            break
        if tecla in (ord('r'), ord('R')):
            panel.reiniciar()
finally:
    cerrar_ventanas()

Apartado Optativo: Reflexión sobre un eje/recta arbitraria

In [42]:
panel_recta = Panel('Controles Eje', 'Reflexión sobre Recta', [
    slider('ang_eje', 'Inclinación recta', -90, 90, 45, lambda v: f'{v}°'),
    slider('offset',  'Desplazamiento eje', -alto, alto, 0, px),
], n_info=5)

V_RECTA = 'Reflexión sobre Recta'
cv.namedWindow(V_RECTA, cv.WINDOW_AUTOSIZE)
colocar(panel_recta, [V_RECTA])

try:
    while True:
        theta = np.radians(panel_recta.valor('ang_eje'))
        d = panel_recta.valor('offset')

        # Centro de la imagen como punto de referencia para el eje
        cx, cy = ancho / 2.0, alto / 2.0 + d

        # Vector unitario a lo largo de la recta y normal
        # Matriz: Trasladar a (cx, cy) -> Rotar theta -> Reflejar en Y (S_y = -1) -> Desrotar -> Destrasladar
        c, s = np.cos(theta), np.sin(theta)
        R = np.float32([[c, -s, 0], [s, c, 0], [0, 0, 1]])
        R_inv = np.float32([[c, s, 0], [-s, c, 0], [0, 0, 1]])
        Ref_x = np.float32([[1, 0, 0], [0, -1, 0], [0, 0, 1]])

        M_linea = mat_traslacion(cx, cy) @ R @ Ref_x @ R_inv @ mat_traslacion(-cx, -cy)
        reflejada = cv.warpAffine(imagen, M_linea[:2, :], size)

        # Dibujar la línea del eje en la visualización
        longitud = max(ancho, alto) * 2
        p1 = (int(cx - c * longitud), int(cy - s * longitud))
        p2 = (int(cx + c * longitud), int(cy + s * longitud))
        
        cv.line(reflejada, p1, p2, B(PAL['acento2']), 2, cv.LINE_AA)

        cv.imshow(V_RECTA, reflejada)
        cv.imshow(panel_recta.ventana, panel_recta.dibujar([
            f"Eje: ang={panel_recta.valor('ang_eje')}° dist={d}px",
            "Ecuación: P' = T·R·S_y·R^t·T^-1·P",
            *[" ".join(f"{v:7.2f}" for v in fila) for fila in M_linea]
        ]))

        tecla = cv.waitKey(30) & 0xFF
        if tecla == 27 or ventana_cerrada(panel_recta.ventana): break
        if tecla in (ord('r'), ord('R')): panel_recta.reiniciar()
finally:
    cerrar_ventanas()